Construction du dataset

In [22]:
import re, csv, io
import numpy as np
import pandas as pd

def load_history(path):
    cols = ["date", "station_code", "capacity", "available_mechanical", "available_electrical",
            "station_name", "station_geo", "operative"]
    txt = open(path, encoding="utf-8").read()
    # sépare les lignes collées : un enregistrement commence par un timestamp
    records = re.split(r"(?<!^)(?<!\n)(?=\d{4}-\d\d-\d\dT\d\d:\d\dZ,)", txt)
    rows = []
    for rec in records:
        for r in csv.reader(io.StringIO(rec.strip())):
            if len(r) == 7:          # ancien format sans station_code
                r.insert(1, None)
            if len(r) == 8:
                rows.append(r)
    df = pd.DataFrame(rows, columns=cols)
    df["date"] = pd.to_datetime(df["date"], utc=True)
    for c in ["capacity", "available_mechanical", "available_electrical"]:
        df[c] = df[c].astype(int)
    return df

df = load_history("/home/porcherleo/Telecom/Telecom_Data_Science/Projet/historique_stations.csv")

# on garde seulement les relevés "lisses" (minutes multiples de 5) et sans doublons
df = df[df["date"].dt.minute % 5 == 0]
df = df.drop_duplicates(["date", "station_name"]).reset_index(drop=True)
df

,date,station_code,capacity,available_mechanical,available_electrical,station_name,station_geo,operative
0,2026-10-16 13:20:00+00:00,16107,35,1,2,Benjamin Godard - Victor Hugo,"48.86598,2.27572",True
1,2026-10-16 13:20:00+00:00,40001,28,9,7,Hôpital Mondor,"48.79892,2.45375",True
2,2026-10-16 13:20:00+00:00,32304,28,1,1,Charcot - Benfleet,"48.87837,2.44052",True
3,2026-10-16 13:20:00+00:00,9020,21,4,1,Toudouze - Clauzel,"48.87930,2.33736",True
4,2026-10-16 13:20:00+00:00,14111,25,2,4,Cassini - Denfert-Rochereau,"48.83753,2.33604",True
...,...,...,...,...,...,...,...,...
72763,2026-10-16 22:30:00+00:00,19018,60,8,5,Cité de la Musique,"48.88880,2.39258",True
72764,2026-10-16 22:30:00+00:00,15056,22,3,4,Place Balard,"48.83640,2.27842",True
72765,2026-10-16 22:30:00+00:00,8004,67,7,1,Malesherbes - Place de la Madeleine,"48.87041,2.32324",True
72766,2026-10-16 22:30:00+00:00,12105,33,23,12,Bercy - Villot,"48.84180,2.37679",True


In [23]:
df["available"] = df["available_mechanical"] + df["available_electrical"]
df

,date,station_code,capacity,available_mechanical,available_electrical,station_name,station_geo,operative,available
0,2026-10-16 13:20:00+00:00,16107,35,1,2,Benjamin Godard - Victor Hugo,"48.86598,2.27572",True,3
1,2026-10-16 13:20:00+00:00,40001,28,9,7,Hôpital Mondor,"48.79892,2.45375",True,16
2,2026-10-16 13:20:00+00:00,32304,28,1,1,Charcot - Benfleet,"48.87837,2.44052",True,2
3,2026-10-16 13:20:00+00:00,9020,21,4,1,Toudouze - Clauzel,"48.87930,2.33736",True,5
4,2026-10-16 13:20:00+00:00,14111,25,2,4,Cassini - Denfert-Rochereau,"48.83753,2.33604",True,6
...,...,...,...,...,...,...,...,...,...
72763,2026-10-16 22:30:00+00:00,19018,60,8,5,Cité de la Musique,"48.88880,2.39258",True,13
72764,2026-10-16 22:30:00+00:00,15056,22,3,4,Place Balard,"48.83640,2.27842",True,7
72765,2026-10-16 22:30:00+00:00,8004,67,7,1,Malesherbes - Place de la Madeleine,"48.87041,2.32324",True,8
72766,2026-10-16 22:30:00+00:00,12105,33,23,12,Bercy - Villot,"48.84180,2.37679",True,35


In [24]:
df.groupby("station_name")["available"].mean().round(2)
# Moyenne des disponibles par station 

station_name
11 Novembre 1918 - 8 Mai 1945           30.12
18 juin 1940 - Buzenval                  6.00
8 Mai 1945 - 10 Juillet 1940            19.40
Abbeville - Faubourg Poissonnière        2.27
Abbé Carton - Plantes                    1.62
                                        ...  
Youri Gagarine - Karl Marx               0.54
Yves Farges - Route d'Enghien            7.69
ZAC des Louvresses                      21.96
Édouard Vaillant - Place Jean Jaurès     2.08
Île de la Jatte                         13.15
Name: available, Length: 1516, dtype: float64

In [25]:
# target = available in 30 minutes
# une ligne à 15h => il faut avoir la stat de available à 15h30

# 2026-10-16T13:45Z - 2026-10-16T13:15Z = 30
## 

t1 = "2026-10-16T13:45Z"
t2 = "2026-10-16T13:15Z"

(pd.Timestamp(t1) - pd.Timestamp(t2)).total_seconds() / 60   # 30.0



30.0

In [26]:
# available de la même station 30 min plus tard (NaN s'il n'y a pas de relevé)
s = df.set_index(["station_name", "date"])["available"]
future = pd.MultiIndex.from_arrays([df["station_name"], df["date"] + pd.Timedelta(minutes=30)])
df["available_in_30"] = s.reindex(future).values
df

,date,station_code,capacity,available_mechanical,available_electrical,station_name,station_geo,operative,available,available_in_30
0,2026-10-16 13:20:00+00:00,16107,35,1,2,Benjamin Godard - Victor Hugo,"48.86598,2.27572",True,3,1.0
1,2026-10-16 13:20:00+00:00,40001,28,9,7,Hôpital Mondor,"48.79892,2.45375",True,16,16.0
2,2026-10-16 13:20:00+00:00,32304,28,1,1,Charcot - Benfleet,"48.87837,2.44052",True,2,1.0
3,2026-10-16 13:20:00+00:00,9020,21,4,1,Toudouze - Clauzel,"48.87930,2.33736",True,5,2.0
4,2026-10-16 13:20:00+00:00,14111,25,2,4,Cassini - Denfert-Rochereau,"48.83753,2.33604",True,6,3.0
...,...,...,...,...,...,...,...,...,...,...
72763,2026-10-16 22:30:00+00:00,19018,60,8,5,Cité de la Musique,"48.88880,2.39258",True,13,NaN
72764,2026-10-16 22:30:00+00:00,15056,22,3,4,Place Balard,"48.83640,2.27842",True,7,NaN
72765,2026-10-16 22:30:00+00:00,8004,67,7,1,Malesherbes - Place de la Madeleine,"48.87041,2.32324",True,8,NaN
72766,2026-10-16 22:30:00+00:00,12105,33,23,12,Bercy - Villot,"48.84180,2.37679",True,35,NaN


Une fois que la variable available a été créé il faut regarder les causes externes de l'utilisation ou non d'un velo.

Problème de transport (métro, bus)  | évenements particuliers (pape, tour de France, fête nationale, on verra le 17) | 